This notebook implements a method to compute the log-probability of an arbitrary sequence given an input sequence using a language model from Hugging Face's transformers library.

https://discuss.huggingface.co/t/computing-log-probability-of-an-arbitrary-sequence-given-another-sequence/70069

In [1]:
# !pip install transformers accelerate torch

Conceptually we want to compute the probability of:
$$P(\text{"ice cream"} \mid \text{"I like "})$$

In [2]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

In [3]:
name = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(name, device_map="auto")

full = tokenizer("I like ice cream", add_special_tokens=False)
print(tokenizer.convert_ids_to_tokens(full["input_ids"]))
# ['I', '▁like', '▁ice', '▁cream']

prefix = tokenizer("I like", add_special_tokens=False)
print(tokenizer.convert_ids_to_tokens(prefix["input_ids"]))
# ['I', '▁like']    # note: no leading space on 'I'

suffix = tokenizer("ice cream", add_special_tokens=False)
print(tokenizer.convert_ids_to_tokens(suffix["input_ids"]))
# ['ice', '▁cream']   # note: no leading space on 'ice'

['I', 'Ġlike', 'Ġice', 'Ġcream']
['I', 'Ġlike']
['ice', 'Ġcream']


In [4]:
name = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(name, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(name, device_map="auto")
inputs = "I like "
outputs = "ice cream"

# Encode input and output
input_tokens = tokenizer.encode(inputs, add_special_tokens=False, return_tensors='pt')
output_tokens = tokenizer.encode(outputs, add_special_tokens=False, return_tensors='pt')

# Concatenate input and output tokens
tokens = torch.cat([input_tokens, output_tokens], dim=1).to(device=model.device)

# Get model predictions for the entire sequence at once
with torch.no_grad():
    outputs = model(tokens)
    logits = outputs.logits

log_sum: float = 0
range_index = range(input_tokens.shape[1] - 1, tokens.shape[1] - 1)
for i in range_index:
    previous_tok, current_tok = i, i + 1
    token_logit = logits[0, previous_tok, :]
    token_log_probs = torch.nn.functional.log_softmax(token_logit, dim=-1)
    log_token_prob = token_log_probs[tokens[0, current_tok]].item()
    log_sum += log_token_prob

    token = tokenizer.decode(tokens[:, current_tok])
    print(f"Token: {token}, Log Prob: {log_token_prob}")

print(f"Total Log Sum Probability: {log_sum}")

Token: ice, Log Prob: -13.381272315979004
Token:  cream, Log Prob: -0.4122574031352997
Total Log Sum Probability: -13.793529719114304


In [5]:
name = "Qwen/Qwen2.5-1.5B"
tokenizer = AutoTokenizer.from_pretrained(name, device_map="auto")
model = AutoModelForCausalLM.from_pretrained(name, device_map="auto")
inputs = "I like "
outputs = "ice cream"

input_tokens = tokenizer.encode(inputs, add_special_tokens=False, return_tensors="pt").to(device=model.device)
output_tokens = tokenizer.encode(outputs, add_special_tokens=False, return_tensors="pt").to(device=model.device)
input_tokens_updated = input_tokens.clone()
log_sum = 0

for i in range(output_tokens.shape[1]):
    # Predict with the given model
    with torch.no_grad():
        outputs = model(input_tokens_updated)
        logit_predictions = outputs.logits

    # Extract the log probability of the most recently added token
    last_token_logit = logit_predictions[0, -1, :]
    last_token_log_probs = torch.nn.functional.log_softmax(last_token_logit, dim=-1)
    log_token_prob = last_token_log_probs[output_tokens[0, i]].item()
    log_sum += log_token_prob

    # Incrementally add an output token to the current sequence
    last_token = tokenizer.decode(output_tokens[:, i])
    input_tokens_updated = torch.cat([input_tokens_updated, output_tokens[:, i:i+1]], dim=1)
    print([tokenizer.decode(token) for token in input_tokens_updated])
    print(f"Token: {last_token}, Log Prob: {log_token_prob}")
print(f"Total Log Sum Probability: {log_sum}")

['I like ice']
Token: ice, Log Prob: -13.381272315979004
['I like ice cream']
Token:  cream, Log Prob: -0.4122574031352997
Total Log Sum Probability: -13.793529719114304
